# Preprocessing

Preprocessing the disaster and finance data into a unified format.

In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np

from climatefinance import preprocessing, utils

In [2]:
# Read data
disaster_data = utils.get_disaster_data()
finance_master = utils.get_finance_data()

Loaded 19331 rows from data/disaster/disaster_dataset_cleaned.dta
Loaded 73899 rows from data/finance/finance_disaster_master.dta


In [3]:
disaster_data.head()

,fips,time,event_id,event_type,tor_f_scale,injuries_indirect,deaths_direct,deaths_indirect,property_damage_value,damage_property,damage_crops,injuries_direct,event_narrative
0,1001,1998-12-01,5675197,Winter Wave,,0,0,0,2000000.0,2M,0K,0,
1,1001,2003-04-01,5354304,Hail,,0,0,0,2500000.0,2.5M,0K,0,Supercell 3 continued on its east southeast mo...
2,1001,2003-04-01,5354305,Thunderstorm,,0,0,0,1000000.0,1M,0K,3,Supercell 3 also produced an area of significa...
3,1001,2004-11-01,5425733,Tornado,F2,0,0,0,900000.0,900K,,1,
4,1001,2008-02-01,85070,Tornado,EF3,0,0,0,10000000.0,10.00M,0.00K,50,The tornado touched down near the waste water ...


In [4]:
finance_master.head()

,State,County,fips,time,Coastal_County,Early_Delinquency_Rate,Late_Delinquency_Rate,Event_Occur,Event_Id,Event_Type,...,Deaths_Indirect,Property_Damage_Value,Damage_Property,Damage_Crops,Injuries_Direct,Event_Narrative,Land_Area,Coastline_Region,Pop,Distance_To_Coast
0,AL,Baldwin County,1003,2008-01-01,Coastal,2.8,1.3,0.0,NaN,,...,NaN,NaN,,,NaN,,1589.780029,Gulf of Mexico,208563.0,21.3906
1,AL,Baldwin County,1003,2008-02-01,Coastal,3.1,1.2,0.0,NaN,,...,NaN,NaN,,,NaN,,1589.780029,Gulf of Mexico,208563.0,21.3906
2,AL,Baldwin County,1003,2008-03-01,Coastal,3.2,1.1,0.0,NaN,,...,NaN,NaN,,,NaN,,1589.780029,Gulf of Mexico,208563.0,21.3906
3,AL,Baldwin County,1003,2008-04-01,Coastal,2.5,1.3,0.0,NaN,,...,NaN,NaN,,,NaN,,1589.780029,Gulf of Mexico,208563.0,21.3906
4,AL,Baldwin County,1003,2008-05-01,Coastal,2.7,1.5,0.0,NaN,,...,NaN,NaN,,,NaN,,1589.780029,Gulf of Mexico,208563.0,21.3906


## Basic data cleaning

The following cells standardize both datasets into a common panel format and run structural diagnostics.

1. **`prepare_monthly_panel`** — Parses the `time` column to datetime, zero-pads `fips` codes to 5 characters (preserving leading zeros), and adds a `month` column truncated to month-start. Applied identically to both datasets so they share a common join key.

2. **`panel_diagnostics`** — Inspects each dataset for: number of unique counties and time periods, duplicate county-month rows, whether the panel is balanced (every county in every month), and which counties have gaps. This tells us whether the data is merge-ready.

3. **`disaster_county_month_check`** — Counts how many disaster events fall in the same county-month and surfaces the top co-occurring examples. This confirms that the disaster data is event-level (multiple rows per county-month) and must be aggregated before merging.

In [5]:
# Prepare both datasets
finance_master_clean = preprocessing.prepare_monthly_panel(
    finance_master, time_col="time", fips_col="fips"
)
disaster_data_clean = preprocessing.prepare_monthly_panel(
    disaster_data, time_col="time", fips_col="fips"
)

In [6]:
# Finance panel diagnostics
finance_diag = preprocessing.panel_diagnostics(
    finance_master_clean,
    fips_col="fips",
    month_col="month",
    name="finance_master"
)


Diagnostics for: finance_master
Unique counties: 357
Unique time periods: 207
Date range: 2008-01-01 00:00:00 to 2025-03-01 00:00:00
Duplicate county-month rows: 0
Balanced panel? True
Observation counts per county:
count    357.0
mean     207.0
std        0.0
min      207.0
25%      207.0
50%      207.0
75%      207.0
max      207.0
Name: month, dtype: float64
Total missing county-month combinations: 0


In [7]:
# Disaster panel diagnostics
disaster_diag = preprocessing.panel_diagnostics(
    disaster_data_clean,
    fips_col="fips",
    month_col="month",
    name="disaster_data"
)


Diagnostics for: disaster_data
Unique counties: 2879
Unique time periods: 512
Date range: 1980-02-01 00:00:00 to 2025-02-01 00:00:00
Duplicate county-month rows: 2195
Top duplicate county-month combinations:
        fips      month  n_rows
12388  45019 2015-10-01      25
13543  48113 2012-06-01      17
14469  48439 2021-04-01      16
9520   36007 2011-09-01      15
12504  45079 2015-10-01      14
14459  48439 2009-03-01      14
6292   26017 2014-07-01      14
14339  48375 2013-05-01      13
13583  48121 2023-06-01      12
15966  55123 2008-06-01      12
Balanced panel? False
Observation counts per county:
count    2879.000000
mean        5.582494
std         4.500411
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        46.000000
Name: month, dtype: float64
Total missing county-month combinations: 1541467
Counties with most missing months:
fips
16045    540
47069    540
47089    540
13011    540
31127    540
31113    540
13003    540
13001    5

In [8]:
# Check multiple disasters in same county-month
disaster_repeat = preprocessing.disaster_county_month_check(
    disaster_data_clean,
    fips_col="fips",
    month_col="month",
    event_col="event_type"
)


Disaster county-month repetition check
County-months with >1 disaster row: 2195
Top county-months with multiple disasters:
        fips      month  n_disasters
12388  45019 2015-10-01           25
13543  48113 2012-06-01           17
14469  48439 2021-04-01           16
9520   36007 2011-09-01           15
14459  48439 2009-03-01           14
12504  45079 2015-10-01           14
6292   26017 2014-07-01           14
14339  48375 2013-05-01           13
6525   26103 2007-06-01           12
13583  48121 2023-06-01           12

Example event types in repeated county-months:
      fips      month  n_disasters event_type
101  26017 2014-07-01           14       Hail
102  26017 2014-07-01           14       Hail
103  26017 2014-07-01           14       Hail
104  26017 2014-07-01           14       Hail
105  26017 2014-07-01           14       Hail
..     ...        ...          ...        ...
53   48439 2021-04-01           16       Hail
54   48439 2021-04-01           16       Hail
55   48

### Basic Data Cleaning - Findings

We observe that finance_master is clean and model-ready, on the other hand disaster_data is not a panel in the same sense, we have several duplicate dates, or counties-months with more than one-disaster row.

The correct workflow is to restrict disasters to:
- counties in finance_master
- months in finance_master
- selected disaster categories
- convert event-level rows into county-month summaries
- merge the aggregated disaster table into finance_master

In [9]:
# Restrict disasters
finance = finance_master_clean.copy()
disasters = disaster_data_clean.copy()

finance_counties = set(finance["fips"].unique())
finance_min_month = finance["month"].min()
finance_max_month = finance["month"].max()

# We are selecting only the rows where counti-month are also in finance_master data set
disasters_sub = disasters[
    disasters["fips"].isin(finance_counties) &
    disasters["month"].between(finance_min_month, finance_max_month)
].copy()

print("Disaster rows after restricting to finance counties and months:", len(disasters_sub))
print("Unique counties after restriction:", disasters_sub["fips"].nunique())
print("Date range:", disasters_sub["month"].min(), "to", disasters_sub["month"].max())

Disaster rows after restricting to finance counties and months: 1556
Unique counties after restriction: 329
Date range: 2008-01-01 00:00:00 to 2025-02-01 00:00:00


In [10]:
# Parse damages
disasters_sub["damage_property_num"] = (
    disasters_sub["damage_property"].apply(preprocessing.parse_damage)
    if "damage_property" in disasters_sub.columns
    else np.nan
)

disasters_sub["damage_crops_num"] = (
    disasters_sub["damage_crops"].apply(preprocessing.parse_damage)
    if "damage_crops" in disasters_sub.columns
    else 0.0
)

disasters_sub["damage_property_num"] = disasters_sub["damage_property_num"].fillna(0)
disasters_sub["damage_crops_num"] = disasters_sub["damage_crops_num"].fillna(0)

disasters_sub["total_damage"] = (
    disasters_sub["damage_property_num"] + disasters_sub["damage_crops_num"]
)

# Standardize event types
disasters_sub["event_type_grouped"] = disasters_sub["event_type"].apply(
    preprocessing.normalize_event_type
)

# Quick check
print(disasters_sub["event_type_grouped"].value_counts(dropna=False))

event_type_grouped
Flood             706
Tornado           346
Thunderstorm      229
Hail              132
Other              49
Tropical Storm     43
Hurricane          33
Wildfire           18
Name: count, dtype: int64


In [11]:
# Restrict our data to some disaster events and some threshold of damages
target_types = utils.get_target_types()

disasters_filtered = disasters_sub[
    disasters_sub["event_type_grouped"].isin(target_types)
].copy()

# Baseline threshold
disasters_filtered_500k = disasters_filtered[
    disasters_filtered["total_damage"] >= 500_000
].copy()

print("Rows after type filter:", len(disasters_filtered))
print("Rows after type + 500k threshold:", len(disasters_filtered_500k))
print(disasters_filtered_500k["event_type_grouped"].value_counts())

Rows after type filter: 1507
Rows after type + 500k threshold: 1507
event_type_grouped
Flood             706
Tornado           346
Thunderstorm      229
Hail              132
Tropical Storm     43
Hurricane          33
Wildfire           18
Name: count, dtype: int64


In [12]:
# Main county-month aggregation
cols = disasters_filtered_500k.columns

disaster_monthly = (
    disasters_filtered_500k
    .groupby(["fips", "month"], as_index=False)
    .agg(
        n_disasters=("event_type_grouped", "size"),
        total_damage=("total_damage", "sum"),
        max_damage=("total_damage", "max"),
        n_hurricane=("event_type_grouped", lambda s: (s == "Hurricane").sum()),
        n_tornado=("event_type_grouped", lambda s: (s == "Tornado").sum()),
        n_tropical_storm=("event_type_grouped", lambda s: (s == "Tropical Storm").sum()),
        n_thunderstorm=("event_type_grouped", lambda s: (s == "Thunderstorm").sum()),
        n_flood=("event_type_grouped", lambda s: (s == "Flood").sum()),
        n_winter_weather=("event_type_grouped", lambda s: (s == "Winter Weather").sum()),
        n_wildfire=("event_type_grouped", lambda s: (s == "Wildfire").sum()),
        n_hail=("event_type_grouped", lambda s: (s == "Hail").sum()),
        injuries_direct=(
            ("injuries_direct", "sum") if "injuries_direct" in cols
            else ("event_type_grouped", "size")
        ),
        injuries_indirect=(
            ("injuries_indirect", "sum") if "injuries_indirect" in cols
            else ("event_type_grouped", "size")
        ),
        deaths_direct=(
            ("deaths_direct", "sum") if "deaths_direct" in cols
            else ("event_type_grouped", "size")
        ),
        deaths_indirect=(
            ("deaths_indirect", "sum") if "deaths_indirect" in cols
            else ("event_type_grouped", "size")
        ),
    )
)

disaster_monthly["event_occur"] = 1
disaster_monthly["log_total_damage"] = np.log1p(disaster_monthly["total_damage"])

print(disaster_monthly.head())
print("County-month rows in aggregated disaster table:", len(disaster_monthly))

    fips      month  n_disasters  total_damage   max_damage  n_hurricane  \
0  01003 2014-04-01            1    27000000.0   27000000.0            0   
1  01003 2017-01-01            2     1550000.0     950000.0            0   
2  01073 2008-02-01            1     1000000.0    1000000.0            0   
3  01073 2011-04-01            3   723300000.0  700000000.0            0   
4  01073 2023-12-01            1     5000000.0    5000000.0            0   

   n_tornado  n_tropical_storm  n_thunderstorm  n_flood  n_winter_weather  \
0          0                 0               0        1                 0   
1          0                 0               1        1                 0   
2          1                 0               0        0                 0   
3          3                 0               0        0                 0   
4          1                 0               0        0                 0   

   n_wildfire  n_hail  injuries_direct  injuries_indirect  deaths_direct  \
0   

In [13]:
# Merging the data sets
analysis_df = finance.merge(
    disaster_monthly,
    on=["fips", "month"],
    how="left"
)

In [14]:
# Inputation

# Fill untreated county-months with zeros
fill_zero_cols = [
    "n_disasters", "total_damage", "max_damage", "event_occur", "log_total_damage",
    "n_hurricane", "n_tornado", "n_tropical_storm", "n_thunderstorm",
    "n_flood", "n_winter_weather", "n_wildfire", "n_hail",
    "injuries_direct", "injuries_indirect", "deaths_direct", "deaths_indirect"
]

for col in fill_zero_cols:
    if col in analysis_df.columns:
        analysis_df[col] = analysis_df[col].fillna(0)

print(analysis_df.shape)
print("Share of treated county-months:", analysis_df["event_occur"].mean())

(73899, 42)
Share of treated county-months: 0.01575122802744286


In [15]:
# Save the final dataset
utils.save_analysis_data(analysis_df, index=False)

Saved 73899 rows to data/analysis/finance_disaster_analysis.csv
